# Data–MC overlays — PRL Product A & B

Headless driver: [`../scripts/data_mc_overlay_products.py`](../scripts/data_mc_overlay_products.py)

| Product | Selection | Counts source | Systematics |
|---|---|---|---|
| **B** | `sel_mup` (BQ → **12,804** events) | filled from DFs → `overlay_histdata.pkl` | `productB_sel_mup` CategorySummary **rate** |
| **A** | cut-stage (`sel_all` pipeline) | aggregate `event_selection-batched-live-PRL` → `merged_histdata.pkl` (May 20260525 was pre-DQ ~12967 and is **not** used) | `productA_sel_all` on-the-fly **rate** (no MCstat; cosmics = raw × topology contamination; detector placeholder bins sanitized) |

**Product B data pin:** among χ²μ / FV variants, `2026_09_01_064250__sel_mup-data-1e20-fvfix` is the one with **12,804** events after beam quality (`fom=0.98`, `min_run=20`). Matching MC: `2026_09_01_063924__sel_mup-mc-fvfix`.

**Product A note:** live-PRL batches were filled without the DQ filter, so final-stage data sums are ~12,967. Cut-stage syst bands apply only where PRL Product A packs exist (`nu_score`, `n_trks`, `track_score`, `trk_len`, `vtx_dist`, `mcs_range_diff`, stage-tagged χ²).

Plot style matches `selected_xsec_overlay.ipynb` (topology / genie_sb stacks, ratio panel).

Outputs:
```
/exp/sbnd/data/users/munjung/xsec/numucc_1p0pi/PRL/data_mc_overlays/
  productB_sel_mup/
  productA_sel_all/
```
Each directory has PNGs, cached counts, and `counts_report.npz` (data + MC signal/background bin-by-bin).

In [ ]:
import os

REPO = "/exp/sbnd/app/users/munjung/xsec/freeze/cafpyana"
PY = os.path.join(REPO, "envs/venv_py310_cafpyana/bin/python")
SCRIPT = os.path.join(
    REPO, "analysis_village/numucc_1p0pi/scripts/data_mc_overlay_products.py"
)
OUT = "/exp/sbnd/data/users/munjung/xsec/numucc_1p0pi/PRL/data_mc_overlays"

print(PY)
print(SCRIPT)
print(OUT)

## Run

Prefer the venv Python (newer NumPy can unpickle PRL NPZs). Examples:

```bash
# both products
$PY $SCRIPT --product both

# Product B only (asserts n_evt_good == 12804)
$PY $SCRIPT --product B

# Product A only (aggregate live-PRL batches + overlays)
$PY $SCRIPT --product A

# refill counts caches
$PY $SCRIPT --product both --force-rebuild
```

Or run the next cell (Product B first — validates the 12,804 check).

In [ ]:
import subprocess

# Change to --product both / A as needed. Use --force-rebuild to refill counts.
cmd = [PY, SCRIPT, "--product", "B"]
print(" ".join(cmd), flush=True)
subprocess.check_call(cmd)

In [ ]:
from pathlib import Path
import json

for prod in ("productB_sel_mup", "productA_sel_all"):
    d = Path(OUT) / prod
    man = d / "counts_report_manifest.json"
    print(f"\n=== {prod} ===")
    print(" exists:", d.is_dir(), " manifest:", man.is_file())
    if man.is_file():
        m = json.loads(man.read_text())
        print(" n_variables:", m["n_variables"])
        print(" extra:", m.get("extra"))
        for row in m["variables"][:3]:
            print(
                f"  {row['slug']}: data={row['data_sum']:.1f} "
                f"sig={row['mc_signal_sum']:.1f} bkg={row['mc_background_sum']:.1f}"
            )